# Create BPM datums

In [1]:
from pytao import Tao
import numpy as np
import os

# import logging
# logger = logging.getLogger()
# logger.setLevel(logging.DEBUG)
# logging.debug("test")

In [2]:
# Models to use:

MODELS = ['cu_sxr',
 #'hxr',
 'cu_spec',
 #'lcls_complex',
 'sc_diag0',
 'cu_hxr',
 #'cu_inj',
 #'sc_dasel',
# 'cu_linac',
 'sc_sxr',
 'sc_hxr']

MODELS

['cu_sxr', 'cu_spec', 'sc_diag0', 'cu_hxr', 'sc_sxr', 'sc_hxr']

In [3]:
# Special to remove. These have no electronics
EXCUDE_NAMES = {'BPMDBL1',
 'BPMDL14',
 'BPMDL18',
 'BPME31B',
 'BPME32B',
 'BPME35B',
 'BPME36B',
 'BPMVB2B'}


In [4]:
INITFILE = {model:f'$LCLS_LATTICE/bmad/models/{model}/tao.init' for model in MODELS}
INITFILE

{'cu_sxr': '$LCLS_LATTICE/bmad/models/cu_sxr/tao.init',
 'cu_spec': '$LCLS_LATTICE/bmad/models/cu_spec/tao.init',
 'sc_diag0': '$LCLS_LATTICE/bmad/models/sc_diag0/tao.init',
 'cu_hxr': '$LCLS_LATTICE/bmad/models/cu_hxr/tao.init',
 'sc_sxr': '$LCLS_LATTICE/bmad/models/sc_sxr/tao.init',
 'sc_hxr': '$LCLS_LATTICE/bmad/models/sc_hxr/tao.init'}

In [5]:
# Tack on FACET-II if available
model = 'f2e_inj'
ifile = os.path.expandvars(f'$FACET2_LATTICE/bmad/models/{model}/tao.init')
if os.path.exists(ifile):
    MODELS.append(model)
    INITFILE[model] = ifile

# Utilities

In [6]:
def replace_blocks(text, block_dict, delim='!---'):
    """
    
    Searches for a block of text of the form:
    
    !---
    ! block_name
    
    some old text
    
    !---
    
    and replaces old text with block_text
    
    """
    lines = text.split('\n')
    inside = False
    found=False
    block = {}
    newlines = []
    found = []
    for i, line in enumerate(lines):
        if line.startswith(delim):
            this_delim = line
            if not inside:
                # Just getting inside
                nline = lines[i+1].strip()
                if nline.startswith('!'):
                    key = nline[1:].strip()
                    if key in block_dict:
                        if key in found: 
                            raise ValueError(f'Mutiple blocks found with key: {key}')
                        found.append(key)
                        inside = True
                        
                        block_name = key
                        block_text = block_dict[key]
                        
                        newlines.append( f'{this_delim}\n! {block_name}\n{block_text}\n{this_delim}')
                    else:
                        inside = False
            else:
                # Exiting
                inside = False
                continue
        # Collect regular text
        if not inside:
            newlines.append(line)
    
    for key in block_dict:
        if key not in found:
            raise ValueError(f'Block not found: {key}')
    
    return '\n'.join(newlines)
    
TEXT = """

A 

B


!--------
! Block 1

!---

inside text


!--------
! Block 2

!---


trailing text

"""

res = replace_blocks(TEXT, {'Block 1':'abafag', 'Block 2':'zxxx' })
#for _ in range(10):
#    res = replace_blocks(res, {'some title':'abafag'})
print(res)



A 

B


!--------
! Block 1
abafag
!--------

inside text


!--------
! Block 2
zxxx
!--------


trailing text




In [7]:
def nice_ele_line(eles):
    line = ''
    ii = 0
    for ele in eles:
        ii += len(ele)
        if ii>40:
            ii = 0
            line+=f",\n     '{ele}'"
        else:
            line += f", '{ele}'"
    line = line[1:]
    return line
nice_ele_line(['a', 'b'])

" 'a', 'b'"

# BPMs

These should be hand copy-pasted into `tao.init` files.

In [8]:
# test
tao = Tao(f'-init $LCLS_LATTICE/bmad/models/cu_sxr/tao.init -noplot')
#tao = Tao(f'-init $FACET2_LATTICE/bmad/models/f2e_inj/tao.init -noplot')

In [9]:
# These correspond to orbit.x, orbit.y datums
eles = tao.lat_list('monitor::bpm*,monitor::rfb*', 'ele.name', flags='-no_slaves')
eles[0:5]

['BPM2', 'BPM3', 'BPM5', 'BPM6', 'BPM8']

In [10]:
match='monitor::bpm*,monitor::rfb*'

# Sort by s
eles = tao.lat_list(match, 'ele.name', flags='-array_out -no_slaves')
s = tao.lat_list(match, 'ele.s', flags='-array_out -no_slaves')
eles = np.array(eles)[s.argsort()].tolist()

In [11]:
def make_bpm_datums(tao, match='monitor::bpm*,monitor::rfb*'):
    
    # Sort eles by s
    eles = tao.lat_list(match, 'ele.name', flags='-array_out -no_slaves')
    s = tao.lat_list(match, 'ele.s', flags='-array_out -no_slaves')
    eles = np.array(eles)[s.argsort()].tolist()
    
    eles = [ele for ele in eles if ele not in EXCUDE_NAMES]
    
    
    # 
    line = nice_ele_line(eles)
    data=f"""! Auto-generated BPM datums using: $LCLS_LATTICE/bmad/conversion/tao/create_vars_and_datums.ipynb
        
&tao_d2_data
    d2_data%name = 'orbit'
    universe = 1
    n_d1_data = 3
/    
    
&tao_d1_data
    ix_d1_data = 1
    default_weight = 1
    d1_data%name = 'x'
    default_data_type = 'bpm_orbit.x'
    default_data_source = 'lat'
    !search_for_lat_eles = "{match}" 
    datum(1:)%ele_name = {line}
/

&tao_d1_data
    ix_d1_data = 2
    default_weight = 1
    d1_data%name = 'y'
    default_data_type = 'bpm_orbit.y'
    default_data_source = 'lat'
    !search_for_lat_eles = "{match}" 
    use_same_lat_eles_as = 'orbit.x'
/

&tao_d1_data
    ix_d1_data = 3
    default_weight = 1
    d1_data%name = 'charge'
    default_data_type = 'bunch_charge.live'
    default_data_source = 'beam'
    !search_for_lat_eles = "{match}" 
    use_same_lat_eles_as = 'orbit.x'
/




"""
    return data
    
    
print(make_bpm_datums(tao))

! Auto-generated BPM datums using: $LCLS_LATTICE/bmad/conversion/tao/create_vars_and_datums.ipynb
        
&tao_d2_data
    d2_data%name = 'orbit'
    universe = 1
    n_d1_data = 3
/    
    
&tao_d1_data
    ix_d1_data = 1
    default_weight = 1
    d1_data%name = 'x'
    default_data_type = 'bpm_orbit.x'
    default_data_source = 'lat'
    !search_for_lat_eles = "monitor::bpm*,monitor::rfb*" 
    datum(1:)%ele_name =  'BPM2', 'BPM3', 'BPM5', 'BPM6', 'BPM8', 'BPM9', 'BPM10', 'BPM11', 'BPM12',
     'BPM13', 'BPM14', 'BPM15', 'BPMA11', 'BPMA12', 'BPM21201', 'BPMS11',
     'BPMM12', 'BPM21301', 'BPMM14', 'BPM21401', 'BPM21501', 'BPM21601',
     'BPM21701', 'BPM21801', 'BPM21901', 'BPM22201', 'BPM22301', 'BPM22401',
     'BPM22501', 'BPM22601', 'BPM22701', 'BPM22801', 'BPM22901', 'BPM23201',
     'BPM23301', 'BPM23401', 'BPM23501', 'BPM23601', 'BPM23701', 'BPM23801',
     'BPM23901', 'BPM24201', 'BPM24301', 'BPM24401', 'BPM24501', 'BPM24601',
     'BPM24701', 'BPMS21', 'BPM24901', 'BPM25

# Correctors

In [12]:

def make_corrector_datums(tao):
    
    # Sort eles by s
    xeles = tao.lat_list('hkicker::*', 'ele.name', flags='-array_out -no_slaves')
    yeles = tao.lat_list('vkicker::*', 'ele.name', flags='-array_out -no_slaves')
    
    xline = nice_ele_line(xeles)
    yline = nice_ele_line(yeles)
    
    data=f"""! Auto-generated corrector datums using: $LCLS_LATTICE/bmad/conversion/tao/create_vars_and_datums.ipynb
        
&tao_var
    v1_var%name = 'xcor'
    default_step = 1e-2
    default_attribute = 'bl_kick'
    var(1:)%ele_name = {xline}
/


&tao_var
    v1_var%name = 'ycor'
    default_step = 1e-2
    default_attribute = 'bl_kick'
    var(1:)%ele_name = {yline}
/
"""
    return data

print(make_corrector_datums(tao))

! Auto-generated corrector datums using: $LCLS_LATTICE/bmad/conversion/tao/create_vars_and_datums.ipynb
        
&tao_var
    v1_var%name = 'xcor'
    default_step = 1e-2
    default_attribute = 'bl_kick'
    var(1:)%ele_name =  'XC00', 'XC01', 'XC02', 'XC03', 'XC04', 'XC05', 'XC06', 'XC07', 'XC08', 'XC09',
     'XC10', 'XC21101', 'XC21135', 'XC21165', 'XC21175', 'XC21191',
     'XC21275', 'XC21302', 'XC21325', 'XC21402', 'XC21502', 'XC21602',
     'XC21702', 'XC21802', 'XC21900', 'XC22202', 'XC22302', 'XC22402',
     'XC22502', 'XC22602', 'XC22702', 'XC22802', 'XC22900', 'XC23202',
     'XC23302', 'XC23402', 'XC23502', 'XC23602', 'XC23702', 'XC23802',
     'XC23900', 'XC24202', 'XC24302', 'XC24402', 'XC24502', 'XC24602',
     'XC24702', 'XC24900', 'XC25202', 'XC25302', 'XC25402', 'XC25502',
     'XC25602', 'XC25702', 'XC25802', 'XC25900', 'XC26202', 'XC26302',
     'XC26402', 'XC26502', 'XC26602', 'XC26702', 'XC26802', 'XC26900',
     'XC27202', 'XC27302', 'XC27402', 'XC27502', 'XC276

# Write to file

In [13]:
def replace_tao_init(ifile):
    ifile = os.path.expandvars(ifile)
    tao = Tao(f'-init {ifile} -noplot')    
    blocks = {'BPM Orbit':make_bpm_datums(tao)}
    blocks['Correctors'] = make_corrector_datums(tao)
    
    
    new_itext = replace_blocks(open(ifile).read(), blocks)
    with open(ifile, 'w') as f:
        f.write(new_itext)
    print('Written:', ifile)
  #  print(new_itext)
replace_tao_init('$LCLS_LATTICE/bmad/models/cu_spec/tao.init')

Written: /Users/chrisonian/Code/GitHub/lcls-lattice//bmad/models/cu_spec/tao.init


In [ ]:
for model in MODELS:
    ifile = INITFILE[model]
    print(model, ifile)
    replace_tao_init(ifile)
    

cu_sxr $LCLS_LATTICE/bmad/models/cu_sxr/tao.init
Written: /Users/chrisonian/Code/GitHub/lcls-lattice//bmad/models/cu_sxr/tao.init
cu_spec $LCLS_LATTICE/bmad/models/cu_spec/tao.init
Written: /Users/chrisonian/Code/GitHub/lcls-lattice//bmad/models/cu_spec/tao.init
sc_diag0 $LCLS_LATTICE/bmad/models/sc_diag0/tao.init
Written: /Users/chrisonian/Code/GitHub/lcls-lattice//bmad/models/sc_diag0/tao.init
cu_hxr $LCLS_LATTICE/bmad/models/cu_hxr/tao.init
Written: /Users/chrisonian/Code/GitHub/lcls-lattice//bmad/models/cu_hxr/tao.init
sc_sxr $LCLS_LATTICE/bmad/models/sc_sxr/tao.init
